# B2-019-attention-transformers — Practice p24 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** transformer-residual-layernorm, position-wise-feed-forward, transformer-block

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

The original applies the mask after softmax, normalizes after residual additions (post-norm rather than pre-norm), feeds FFN from x rather than the first residual output y, and leaves attention's query/key/value source ambiguous. A corrected encoder block is: a=LN1(x); attention_output,weights=attention(a,a,a,allowed_mask_before_softmax); y=x+attention_output; f=LN2(y); z=y+FFN(f). Forbidden weights must be exactly zero, a zero attention gives y=x and a zero FFN gives z=y, and every intermediate preserves (B,N,D).

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-12
RTOL = 1e-12
B, N, D = 2, 3, 4
x = np.arange(B * N * D, dtype=np.float64).reshape(B, N, D) / 10.0
allowed = np.tril(np.ones((N, N), dtype=bool))
scores = np.zeros((B, N, N), dtype=np.float64)
masked_scores = np.where(allowed, scores, -np.inf)
shifted = masked_scores - np.max(masked_scores, axis=-1, keepdims=True)
weights = np.exp(shifted)
weights = weights / np.sum(weights, axis=-1, keepdims=True)

def layer_norm_rows(value):
    mean = np.mean(value, axis=-1, keepdims=True)
    variance = np.mean((value - mean) ** 2, axis=-1, keepdims=True)
    return (value - mean) / np.sqrt(variance + 1e-5)

a = layer_norm_rows(x)
attention_output = np.zeros_like(x)
y = x + attention_output
f = layer_norm_rows(y)
ffn_output = np.zeros_like(y)
z = y + ffn_output

### Answer check

In [ ]:
assert np.count_nonzero(weights[:, ~allowed]) == 0
np.testing.assert_allclose(weights.sum(axis=-1), np.ones((B, N)), atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(y, x, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(z, y, atol=ATOL, rtol=RTOL)
assert a.shape == y.shape == f.shape == z.shape == (B, N, D)